# 데이터 병합 과정
1. 등고선 데이터와 들락날락 위치 데이터를 불러옵니다.
2. 들락날락 위치에서 가장 가까운 등고선의 등고수치를 구해서 새로운 컬럼에 추가합니다.
3. 도보로 약 10분 거리인 540m를 기준으로 원을 만들고 원의 경계에 걸치는 등고선을 추출합니다.
4. 추출한 등고선의 등고수치의 최대값과 최소값을 구합니다.
5. 최대값 - 현재 들락날락의 고도 = high_up, 현재 들락날락의 고도 - 최소값 = high_down을 구한 후 새로운 컬럼에 추가합니다.
6. 시군구별 0~12세 인구 비율이 포함된 행정경계 데이터를 가져옵니다.
7. 들락날락의 각 위치가 어느 시군구에 포함되는지 공간 조인하여 시군구명과 아동인구수비율 정보를 추가합니다.
8. 최종적으로 들락날락의 이름과 주소, 해당하는 시군구, 좌표, 어린이 비율, 등고수치가 포함된 데이터프레임을 shp 파일로 저장합니다.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import math

# 등고선 데이터(라인) 읽기 및 좌표계 변환 (EPSG:4326)
contour_gdf = gpd.read_file("../../Data_list/preprocessing_result/부산광역시_등고선_필터링/부산광역시_등고선_전체.gpkg").to_crs(epsg=4326)

# 위치 정보가 담긴 CSV 파일을 읽어 DataFrame 생성
df = pd.read_csv('../../Data_list/preprocessing_result/들락날락_위치_좌표_추출_복사본/부산광역시_좌표추가 복사본.csv')

# 위도/경도 정보를 Point로 변환하여 GeoDataFrame 생성
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['경도'], df['위도']),
    crs='EPSG:4326'
)

# 각 위치별로 가장 가까운 등고선의 등고수치(고도) 구하기
def find_nearest_contour(point, contour_gdf):
    distances = contour_gdf.geometry.distance(point)
    idx_min = distances.idxmin()
    return contour_gdf.loc[idx_min, '등고수치']

gdf['contour'] = gdf.geometry.apply(lambda x: find_nearest_contour(x, contour_gdf))

# 거리 계산을 위해 투영 좌표계(EPSG:5179)로 변환
contour_gdf_proj = contour_gdf.to_crs(epsg=5179)
gdf_proj = gdf.to_crs(epsg=5179)

# 각 위치별로 반경 540m 내 등고선의 등고수치 최소/최대값 및 고도차 계산
gdf['contour_min_540'] = None
gdf['contour_max_540'] = None
gdf['high_up'] = None    # 현재 위치 고도 - 540m 내 등고수치 최소값
gdf['high_down'] = None  # 540m 내 등고수치 최대값 - 현재 위치 고도

for idx, row in gdf_proj.iterrows():
    buffer = row.geometry.buffer(540)
    intersected = contour_gdf_proj[contour_gdf_proj.intersects(buffer)]
    if not intersected.empty:
        z_min = intersected['등고수치'].min()
        z_max = intersected['등고수치'].max()
        z_cur = gdf.loc[idx, 'contour']
        high_up = z_cur - z_min
        high_down = z_max - z_cur
    else:
        z_min, z_max, high_up, high_down = None, None, None, None

    gdf.at[idx, 'contour_min_540'] = z_min
    gdf.at[idx, 'contour_max_540'] = z_max
    gdf.at[idx, 'high_up'] = high_up
    gdf.at[idx, 'high_down'] = high_down

# 시군구별 0~12세 인구 비율이 포함된 행정경계 데이터 읽기
sgg = gpd.read_file('../../Data_list/preprocessing_result/아동인구수전처리결과/시군구별_어린이_비율.shp')

# 좌표계 통일 (EPSG:4326)
points = gdf.to_crs(epsg=4326)
sgg = sgg.to_crs(epsg=4326)

# 각 위치(Point)가 어느 시군구(Polygon)에 포함되는지 공간 조인하여 시군구명과 아동인구수비율 정보 추가
joined = gpd.sjoin(points, sgg[['SGG_NM', 'geometry', '0~12세_비']], how='left', predicate='within')

# 불필요한 컬럼 제거 및 컬럼명 한글로 변경
joined.drop(['주소번호', 'index_right', '위도', '경도'], axis=1, inplace=True)
joined = joined.rename(columns={'SGG_NM': '시군구', '0~12세_비': '어린이비율'})

# 컬럼 순서 재정렬
new_columns = ['이름', '주소', '시군구', 'geometry', '어린이비율', 'contour', 'contour_min_540', 'contour_max_540', 'high_up', 'high_down']
joined = joined[new_columns]

# 결과를 shp 파일로 저장
joined.to_file('../../Data_list/preprocessing_result/들락날락_등고수치_인구병합결과/위치_등고수치_시군구_어린이비율.shp', encoding='utf-8')

# 등고수치 기준 내림차순으로 정렬하고 결과 확인
joined_sorted = joined.sort_values(by='contour', ascending=False)
joined_sorted.head(10)

C:\Users\tjral\AppData\Local\Temp\ipykernel_11780\913343065.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = contour_gdf.geometry.distance(point)
C:\Users\tjral\AppData\Local\Temp\ipykernel_11780\913343065.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = contour_gdf.geometry.distance(point)
C:\Users\tjral\AppData\Local\Temp\ipykernel_11780\913343065.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = contour_gdf.geometry.distance(point)
C:\Users\tjral\AppData\Local\Temp\ipykernel_11780\913343065.py:21: UserWarning: Geometry is

,이름,주소,시군구,geometry,어린이비율,contour,contour_min_540,contour_max_540,high_up,high_down
69,금정아이숲 들락날락 어린이복합문화공간,금정구 산성로 435,부산광역시 금정구,POINT (129.05476 35.2513),0.061106,300.0,250.0,430.0,50.0,130.0
10,서구 숲속놀이터 들락날락,서구 꽃마을로163번길 73(구덕문화공원 내),부산광역시 서구,POINT (129.00527 35.12641),0.061472,280.0,125.0,495.0,155.0,215.0
42,만덕종합사회복지관 들락날락,"북구 함박봉로 140번길 102, 만덕종합사회복지관 4층 들락날락",부산광역시 북구,POINT (129.03539 35.20183),0.075446,185.0,85.0,365.0,100.0,180.0
20,시랑골 아이누리 작은도서관 들락날락,부산광역시 북구 시랑로185번길 47,부산광역시 북구,POINT (129.01546 35.19702),0.075446,180.0,80.0,485.0,100.0,305.0
21,상학도서관 들락날락,북구 상학산복길215,부산광역시 북구,POINT (129.04021 35.21716),0.075446,165.0,75.0,325.0,90.0,160.0
77,서구아미드림도서관 들락날락(조성중),부산시 서구 아미동2가 249-18일원,부산광역시 서구,POINT (129.00922 35.10373),0.061472,140.0,60.0,240.0,80.0,100.0
64,안데르센 이야기관,부산광역시 기장군 장안읍 장안로 211,부산광역시 기장군,POINT (129.23989 35.36014),0.113239,140.0,55.0,225.0,85.0,85.0
98,부산어린이해양복합문화공간 들락날락(조성중),부산광역시 부산진구 초읍동 43번지 부산어린이대공원 관리사업소,부산광역시 부산진구,POINT (129.04169 35.18585),0.071018,120.0,70.0,215.0,50.0,95.0
50,반여도서관 들락날락,해운대구 재반로282번길 38(반여동),부산광역시 해운대구,POINT (129.13381 35.20203),0.084278,120.0,40.0,250.0,80.0,130.0
87,금정도서관(조성중),부산광역시 금정구 금정도서관로 33(청룡동),부산광역시 금정구,POINT (129.0962 35.27272),0.061106,105.0,55.0,190.0,50.0,85.0
